## Google Ads

#### Useful links:
- [Get a Developer account](https://developers.facebook.com/async/registration/dialog/?src=default)
- [Google Ads API Documentation](https://www.facebook.com/ads/library/api/)
- [Pandas GBQ Documentation](https://pandas-gbq.readthedocs.io/en/latest/)
- [Google Ads Transparency](https://adstransparency.google.com/)
- [API Data Dictionary - README](https://storage.googleapis.com/ads-transparency-center/api-data/README.txt)

In [ ]:
# requirements
%pip install pandas requests python-dotenv pandas-gbq tqdm

In [1]:
import os
import json
import pandas as pd
import pandas_gbq
import pickle
import requests
from dotenv import load_dotenv
from datetime import datetime, timezone
from time import sleep

In [2]:
# environment variables
load_dotenv()
PROJECT_ID = os.getenv("PROJECT_ID")

This notebook assumes that no authentication other than project_id is necessary. If needed, please refer to [Pandas GBQ - Authentication](https://pandas-gbq.readthedocs.io/en/latest/howto/authentication.html#default-authentication-methods)

#### Google Ads Tables

**creative_stats:** contains information about advertisers that served ads in the European Economic Area or Turkey: their legal name, verification status, disclosed name, and location. It also includes ad specific information: impression ranges per region (including aggregate impressions for the European Economic Area), first shown and last shown dates, which criteria were used in audience selection, the format of the ad, the ad topic and whether the ad is funded by Google Ad Grants program. A link to the ad in the Google Ads Transparency Center is also provided.

**The removed_creative_stats:** contains information about ads that served in the European Economic Area that Google removed: where and why they were removed and per-region information on when they served. The removed_creative_stats table also contains a link to the Google Ads Transparency Center for the removed ad.

Both datasets are also available for download in [Google Storage](https://console.cloud.google.com/storage/browser/ads-transparency-center/api-data;tab=objects?prefix=&forceOnObjectsSortingFiltering=false)

In [3]:
creative_stats_tb = "bigquery-public-data.google_ads_transparency_center.creative_stats"
removed_creative_stats_tb = "bigquery-public-data.google_ads_transparency_center.removed_creative_stats"

In [4]:
sql_query_example = """
SELECT
  COUNT(*) AS line_count
FROM
  `bigquery-public-data.google_ads_transparency_center.creative_stats` AS CS,
  CS.region_stats AS RS
WHERE
  RS.region_code = "BR" ;
"""

#### API Connection

In [5]:
def query_api(project_id, select):
    return pandas_gbq.read_gbq(select, project_id=project_id)

### Let's sample some lines from the dataset tables
As a "SELECT *" query can be quite costly in terms of our monthly free quota for consulting public BigQuery datasets, to refrain from making unnecessary BigQuery requests, let's get samples from both publicly available creative stats tables and use them to answer questions where possible.

When dealing with questions that need specific types of requests (specially when multiple consecutive requests are needed), it is advised to use a more strict SELECT statement, to consume less quota points.

In [6]:
# Get 100 lines of creative stats on France
sql_query_fr_cs = f"""
SELECT
  CS.*
FROM
  {creative_stats_tb} AS CS
CROSS JOIN
  UNNEST(CS.region_stats) AS RS
WHERE
  RS.region_code = "FR"
LIMIT
  100;
"""

df_creative_stats = query_api(project_id=PROJECT_ID, select=sql_query_fr_cs)
df_creative_stats

Downloading: 100%|█████████████████████████████████████████████████████████████|


,advertiser_id,creative_id,creative_page_url,ad_format_type,advertiser_disclosed_name,advertiser_legal_name,advertiser_location,advertiser_verification_status,region_stats,audience_selection_approach_info,topic,is_funded_by_google_ad_grants,ad_funded_by
0,AR02936389872358785025,CR04002958534038781953,https://adstransparency.google.com/advertiser/...,VIDEO,Pin,,,UNVERIFIED,"[{'region_code': 'DE', 'first_shown': '2025-01...","{'demographic_info': 'CRITERIA_UNUSED', 'geo_l...",Commercial,False,
1,AR02299254993238097921,CR03457172953559465985,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-1...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Business & Industrial,False,"Balboa Centria Group, S.A."
2,AR08652840297924919297,CR03793810806592765953,https://adstransparency.google.com/advertiser/...,TEXT,Daima TIC Solucions SL,Daima TIC Solucions SL,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Apparel,False,
3,AR07498723778797502465,CR14996880020616511489,https://adstransparency.google.com/advertiser/...,VIDEO,"EDI BUSINESS SCHOOL, SLU","EDI BUSINESS SCHOOL, SLU",AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Jobs & Education,False,
4,AR10967695351116464129,CR11025104684313477121,https://adstransparency.google.com/advertiser/...,TEXT,HILL SMART,HILL SMART,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Dining & Nightlife,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,AR12551884589318012929,CR07377182285994393601,https://adstransparency.google.com/advertiser/...,VIDEO,Outscale company FZCO,Outscale company FZCO,AE,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2024-1...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,Health,False,
96,AR12551884589318012929,CR00428861125979799553,https://adstransparency.google.com/advertiser/...,IMAGE,Outscale company FZCO,Outscale company FZCO,AE,VERIFIED,"[{'region_code': 'BE', 'first_shown': '2025-08...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,Business & Industrial,False,
97,AR12551884589318012929,CR04733439510816227329,https://adstransparency.google.com/advertiser/...,IMAGE,Outscale company FZCO,Outscale company FZCO,AE,VERIFIED,"[{'region_code': 'BE', 'first_shown': '2023-08...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Sports & Fitness,False,
98,AR06636938315977195521,CR00518828596203094017,https://adstransparency.google.com/advertiser/...,VIDEO,PLANET BEYOND INFORMATION TECHNOLOGY CO. L.L.C,PLANET BEYOND INFORMATION TECHNOLOGY CO. L.L.C,AE,VERIFIED,"[{'region_code': 'BG', 'first_shown': '2025-10...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Computers & Consumer Electronics,False,


In [7]:
# Get 100 lines of removed creative stats on France
sql_query_fr_rcs = f"""
SELECT
  CS.*
FROM
  {removed_creative_stats_tb} AS CS
CROSS JOIN
  UNNEST(CS.region_stats) AS RS
WHERE
  RS.region_code = "FR"
LIMIT
  100;
"""

df_removed_creative_stats = query_api(project_id=PROJECT_ID, select=sql_query_fr_rcs)
df_removed_creative_stats

Downloading: 100%|█████████████████████████████████████████████████████████████|


,creative_page_url,region_stats,audience_selection_approach_info,disapproval
0,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'CZ', 'first_shown': '2025-07...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Account suspension', 'vio..."
1,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ..."
2,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'DE', 'first_shown': '2024-03...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Account suspension', 'vio..."
3,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ..."
4,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ..."
...,...,...,...,...
95,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ..."
96,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'DE', 'first_shown': '2024-12...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Account suspension', 'vio..."
97,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'AT', 'first_shown': '2025-09...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,"[{'removal_reason': 'Account suspension', 'vio..."
98,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ..."


## (Optional)
We might want to save the resulting samples on disk in case we need them later (also to avoid making more "SELECT *" requests to BigQuery).

In [19]:
# Writing on disk
with open(f'sample-fr-cs.pickle', 'wb') as f:
    pickle.dump(df_creative_stats, f)

with open(f'sample-fr-rcs.pickle', 'wb') as f:
    pickle.dump(df_removed_creative_stats, f)

In [19]:
# Reading from disk
with open('sample-fr-cs.pickle', 'rb') as f:
    df_creative_stats = pickle.load(f)

with open('sample-fr-rcs.pickle', 'rb') as f:
    df_removed_creative_stats = pickle.load(f)

### Special Criteria

**SC1: Does the platform provide an API to access its ad repository and extract data on advertising content?**

*This item verifies whether the platform provides an ad repository API with at least one endpoint for programmatically extracting advertising data. Full availability is confirmed when the API returns information on ads across all categories. The assessment should confirm that the endpoint allows the retrieval and storage of ad data without requiring privileged or internal access beyond standard developer registration.*


In [33]:
# One of the data samples we collected
df_creative_stats

,advertiser_id,creative_id,creative_page_url,ad_format_type,advertiser_disclosed_name,advertiser_legal_name,advertiser_location,advertiser_verification_status,region_stats,audience_selection_approach_info,topic,is_funded_by_google_ad_grants,ad_funded_by
0,AR02936389872358785025,CR04002958534038781953,https://adstransparency.google.com/advertiser/...,VIDEO,Pin,,,UNVERIFIED,"[{'region_code': 'DE', 'first_shown': '2025-01...","{'demographic_info': 'CRITERIA_UNUSED', 'geo_l...",Commercial,False,
1,AR02299254993238097921,CR03457172953559465985,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-1...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Business & Industrial,False,"Balboa Centria Group, S.A."
2,AR08652840297924919297,CR03793810806592765953,https://adstransparency.google.com/advertiser/...,TEXT,Daima TIC Solucions SL,Daima TIC Solucions SL,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Apparel,False,
3,AR07498723778797502465,CR14996880020616511489,https://adstransparency.google.com/advertiser/...,VIDEO,"EDI BUSINESS SCHOOL, SLU","EDI BUSINESS SCHOOL, SLU",AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Jobs & Education,False,
4,AR10967695351116464129,CR11025104684313477121,https://adstransparency.google.com/advertiser/...,TEXT,HILL SMART,HILL SMART,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Dining & Nightlife,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,AR12551884589318012929,CR07377182285994393601,https://adstransparency.google.com/advertiser/...,VIDEO,Outscale company FZCO,Outscale company FZCO,AE,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2024-1...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,Health,False,
96,AR12551884589318012929,CR00428861125979799553,https://adstransparency.google.com/advertiser/...,IMAGE,Outscale company FZCO,Outscale company FZCO,AE,VERIFIED,"[{'region_code': 'BE', 'first_shown': '2025-08...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,Business & Industrial,False,
97,AR12551884589318012929,CR04733439510816227329,https://adstransparency.google.com/advertiser/...,IMAGE,Outscale company FZCO,Outscale company FZCO,AE,VERIFIED,"[{'region_code': 'BE', 'first_shown': '2023-08...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Sports & Fitness,False,
98,AR06636938315977195521,CR00518828596203094017,https://adstransparency.google.com/advertiser/...,VIDEO,PLANET BEYOND INFORMATION TECHNOLOGY CO. L.L.C,PLANET BEYOND INFORMATION TECHNOLOGY CO. L.L.C,AE,VERIFIED,"[{'region_code': 'BG', 'first_shown': '2025-10...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",Computers & Consumer Electronics,False,


**SC3: Can data from both active and inactive ads be extracted?**

*This item verifies whether the platform allows the extraction of ad data through either the GUI or the API, from at least one day after publication to at least one year prior. Full availability is confirmed when both active and inactive ad data are delivered across all ad categories. The assessment should test the interface and endpoints to confirm whether both active and inactive ads can be retrieved.*


In [34]:
print(f'Now {datetime.now(tz=timezone.utc).isoformat()}')

# Iterate through the region_stats column to find the max and min dates
all_last_shown_dates = [
    region_stats['last_shown']
    for creative_region_stats in df_creative_stats.region_stats
    for region_stats in creative_region_stats
]

max_date = max(all_last_shown_dates)
min_date = min(all_last_shown_dates)

print(f'Most recent creative show date {max_date}')
print(f'Least recent creative show date {min_date}')

Now 2025-11-24T19:00:04.222776+00:00
Most recent creative show date 2025-11-23
Least recent creative show date 2023-06-06


### ACCESSIBILITY


**OC3: Can the requested data be extracted directly from the ad repository response?**

This item verifies whether the platform ad repository returns structured data on ad creatives and authorship directly in the response, rather than providing a link that redirects to the data. Audiovisual media files and data (e.g., images, videos, and audio) should not be considered when assessing this item. The assessment should examine sample data responses from both the ad repository GUI and API to confirm that the requested public data is included in the returned payload.*


In [35]:
df_creative_stats.loc[0]

advertiser_id                                                  AR02936389872358785025
creative_id                                                    CR04002958534038781953
creative_page_url                   https://adstransparency.google.com/advertiser/...
ad_format_type                                                                  VIDEO
advertiser_disclosed_name                                                         Pin
advertiser_legal_name                                                                
advertiser_location                                                                  
advertiser_verification_status                                             UNVERIFIED
region_stats                        [{'region_code': 'DE', 'first_shown': '2025-01...
audience_selection_approach_info    {'demographic_info': 'CRITERIA_UNUSED', 'geo_l...
topic                                                                      Commercial
is_funded_by_google_ad_grants                         

The BigQuery Google Ads Transparency tables have ads metadata, but they have no fields for the actual content of the ads other than an URL pointing to each ad's page on the Google Ads Transparency Center.

**OC5: Can data from an individual ad be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from a specific advertisement on the ad repository using a unique identifier, rather than relying on search terms or other parameters and filters. The assessment should review the ad repository documentation and test available features to confirm that an individual ad can be retrieved directly by its unique identifier.*



In [36]:
ad_id = 'CR03457172953559465985'

sql_query_individual_ad = f"""
SELECT
  CS.advertiser_id,
  CS.creative_id,
  CS.creative_page_url,
  CS.ad_format_type,
  CS.advertiser_disclosed_name,
  CS.advertiser_legal_name 	
FROM
  {creative_stats_tb} AS CS
WHERE
  CS.creative_id = '{ad_id}'
LIMIT
  100;
"""

df_individual_ad = query_api(project_id=PROJECT_ID, select=sql_query_individual_ad)
df_individual_ad

Downloading: 100%|█████████████████████████████████████████████████████████████|


,advertiser_id,creative_id,creative_page_url,ad_format_type,advertiser_disclosed_name,advertiser_legal_name
0,AR02299254993238097921,CR03457172953559465985,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL


**OC6: Can data from ads served by a specific advertiser be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from ads run by a specific advertiser, via their username or unique identifier. The assessment should review the ad repository documentation and test any available feature to retrieve data from an individual advertiser.*


In [37]:
advertiser = 'AR02299254993238097921'

sql_query_individual_advertiser = f"""
SELECT
  CS.advertiser_id,
  CS.creative_id,
  CS.creative_page_url,
  CS.ad_format_type,
  CS.advertiser_disclosed_name,
  CS.advertiser_legal_name 	
FROM
  {creative_stats_tb} AS CS
WHERE
  CS.advertiser_id = '{advertiser}'
LIMIT
  100;
"""

df_individual_advertiser = query_api(project_id=PROJECT_ID, select=sql_query_individual_advertiser)
df_individual_advertiser

Downloading: 100%|█████████████████████████████████████████████████████████████|


,advertiser_id,creative_id,creative_page_url,ad_format_type,advertiser_disclosed_name,advertiser_legal_name
0,AR02299254993238097921,CR15736934153079226369,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL
1,AR02299254993238097921,CR05702190979524591617,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL
2,AR02299254993238097921,CR14237097488455565313,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL
3,AR02299254993238097921,CR05641693375863193601,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL
4,AR02299254993238097921,CR05824660906818142209,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL
...,...,...,...,...,...,...
95,AR02299254993238097921,CR16543006468366925825,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL
96,AR02299254993238097921,CR03199034836270448641,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL
97,AR02299254993238097921,CR06841065948208693249,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL
98,AR02299254993238097921,CR11071565578937303041,https://adstransparency.google.com/advertiser/...,VIDEO,Centeia SL,Centeia SL


**OC7: Can ad data be retrieved from the platform using search terms?**

*This item verifies whether data on ad campaigns can be retrieved via individual or combined search terms, enabling the creation of a dataset composed of ads that mention those terms. The assessment should test search-related features to confirm that queries using keywords return relevant ad campaign data.*


Since the BigQuery tables for ads transparency have no fields for the actual content of the ads, it is not possible to retrieve data from the platform applying search terms on the content of ads.

**OC8: Does the platform use locale-neutral data representations?**

*This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are provided in a locale-neutral format, or, if that is not possible, whether relevant locale metadata is included. The assessment should review the ad repository documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata.*


In [58]:
df_creative_stats.region_stats.loc[0][0]

{'region_code': 'DE',
 'first_shown': '2025-01-22',
 'last_shown': '2025-01-22',
 'times_shown_end_date': '2025-01-22',
 'times_shown_lower_bound': 0,
 'times_shown_upper_bound': 1000,
 'times_shown_start_date': '2025-01-22',
 'times_shown_availability_date': None,
 'surface_serving_stats': {'surface_serving_stats': array([{'surface': 'PLAY', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'MAPS', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'YOUTUBE', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'SHOPPING', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'SEARCH', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None}],
        dtype=

Dates are provided without time and timezone information.
Also, no currency info is provided.

### COMPLETENESS

**OC9: Does the platform provide data that allows the identification of advertisers who ran ads?**

*This item verifies whether the platform discloses information on the advertisers responsible for the identified ads. The assessment should confirm whether the advertiser’s page name, URL, and unique identifier can be retrieved.*



In [72]:
# Advertiser related columns
df_creative_stats.loc[2][[
    'advertiser_id',
    'advertiser_disclosed_name',
    'advertiser_legal_name',
    'advertiser_location',
    'advertiser_verification_status'
]]

advertiser_id                     AR08652840297924919297
advertiser_disclosed_name         Daima TIC Solucions SL
advertiser_legal_name             Daima TIC Solucions SL
advertiser_location                                   AD
advertiser_verification_status                  VERIFIED
Name: 2, dtype: object

We can retrieve the advertiser's name and ID, but no URL

**OC10: Does the platform provide data on the funders who paid for ads?**

*This item verifies whether the platform provides data on who paid for ad campaigns. The assessment should confirm whether any sponsor information can be retrieved.*


In [74]:
df_creative_stats[['is_funded_by_google_ad_grants', 'ad_funded_by']]

,is_funded_by_google_ad_grants,ad_funded_by
0,False,
1,False,"Balboa Centria Group, S.A."
2,False,
3,False,
4,False,
...,...,...
95,False,
96,False,
97,False,
98,False,


In [75]:
df_creative_stats.is_funded_by_google_ad_grants.value_counts()

is_funded_by_google_ad_grants
False    100
Name: count, dtype: Int64

In [76]:
df_creative_stats.ad_funded_by.value_counts()

ad_funded_by
                                  91
Optimum media direction fz llc     5
Alibaba                            2
Balboa Centria Group, S.A.         1
Digital Eagle Inc                  1
Name: count, dtype: int64

There are fields regarding who paid for ad campaings, but most of the time it seems the value is an empty string. In our 100 samples, only 9 had filled values.

**OC11: Does the platform provide data on the period during which ads were served?**

*This item verifies whether the platform provides data on the days on which ad campaigns ran. The assessment should review ad records to confirm that campaigns include start and end dates (or equivalent temporal markers) covering their active period*

In [82]:
df_creative_stats.loc[3].region_stats[2]

{'region_code': 'IT',
 'first_shown': '2025-08-12',
 'last_shown': '2025-11-22',
 'times_shown_end_date': '2025-08-25',
 'times_shown_lower_bound': 20000,
 'times_shown_upper_bound': 25000,
 'times_shown_start_date': '2025-08-12',
 'times_shown_availability_date': None,
 'surface_serving_stats': {'surface_serving_stats': array([{'surface': 'MAPS', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'YOUTUBE', 'times_shown_upper_bound': 25000, 'times_shown_lower_bound': 20000, 'times_shown_availability_date': None},
         {'surface': 'SHOPPING', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'PLAY', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'SEARCH', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None}],
    

**OC12: Does the platform provide data on user engagement with ads?**

*This item verifies whether the platform provides data on the total number of user interactions with ad campaigns (e.g., likes, comments, shares, clicks). The assessment should review ad records to confirm that engagement metrics are available and clearly linked to each campaign.*


In [84]:
df_creative_stats.loc[3].region_stats[2]

{'region_code': 'IT',
 'first_shown': '2025-08-12',
 'last_shown': '2025-11-22',
 'times_shown_end_date': '2025-08-25',
 'times_shown_lower_bound': 20000,
 'times_shown_upper_bound': 25000,
 'times_shown_start_date': '2025-08-12',
 'times_shown_availability_date': None,
 'surface_serving_stats': {'surface_serving_stats': array([{'surface': 'MAPS', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'YOUTUBE', 'times_shown_upper_bound': 25000, 'times_shown_lower_bound': 20000, 'times_shown_availability_date': None},
         {'surface': 'SHOPPING', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'PLAY', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'SEARCH', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None}],
    

The data has upper and lower bounds for number of times an ad was shown on total and across different services, like Youtube, Google Play, Google Search etc.

**OC13: Does the platform indicate whether ads were placed by verified or unverified advertisers?**

*This item verifies whether the platform clearly indicates whether advertisers were verified at the time their ads were served. The assessment should review ad records to confirm that a verification status field is present.*


In [86]:
df_creative_stats[['advertiser_id', 'creative_id', 'advertiser_verification_status']]

,advertiser_id,creative_id,advertiser_verification_status
0,AR02936389872358785025,CR04002958534038781953,UNVERIFIED
1,AR02299254993238097921,CR03457172953559465985,VERIFIED
2,AR08652840297924919297,CR03793810806592765953,VERIFIED
3,AR07498723778797502465,CR14996880020616511489,VERIFIED
4,AR10967695351116464129,CR11025104684313477121,VERIFIED
...,...,...,...
95,AR12551884589318012929,CR07377182285994393601,VERIFIED
96,AR12551884589318012929,CR00428861125979799553,VERIFIED
97,AR12551884589318012929,CR04733439510816227329,VERIFIED
98,AR06636938315977195521,CR00518828596203094017,VERIFIED


In [87]:
df_creative_stats.advertiser_verification_status.value_counts()

advertiser_verification_status
VERIFIED      93
UNVERIFIED     7
Name: count, dtype: int64

### COMPLIANCE

**OC14: Does the platform flag ads that were removed due to violations of its guidelines or relevant legislation?**

*This item verifies whether the platform indicates when an ad has been moderated. At a minimum, the platform should provide the reason for removal and the date. The assessment should review ad records to confirm that moderated ads are flagged and that the corresponding moderation details are clearly documented.*



In [108]:
# Table for removed ads
df_removed_creative_stats

,creative_page_url,region_stats,audience_selection_approach_info,disapproval
0,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'CZ', 'first_shown': '2025-07...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Account suspension', 'vio..."
1,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ..."
2,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'DE', 'first_shown': '2024-03...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Account suspension', 'vio..."
3,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ..."
4,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ..."
...,...,...,...,...
95,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ..."
96,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'DE', 'first_shown': '2024-12...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Account suspension', 'vio..."
97,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'AT', 'first_shown': '2025-09...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,"[{'removal_reason': 'Account suspension', 'vio..."
98,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ..."


In [122]:
removed_ad = df_removed_creative_stats.loc[0]
removed_ad_last_shown_date = max([rs['last_shown'] for rs in removed_ad.region_stats])
removed_ad_reason = removed_ad.disapproval[0]

{'last_shown_date': removed_ad_last_shown_date, **removed_ad_reason}

{'last_shown_date': '2025-07-21',
 'removal_reason': 'Account suspension',
 'violation_category': 'PROHIBITED_PRACTICES',
 'use_of_automated_means': False,
 'removal_location': array(['GLOBAL'], dtype=object),
 'decision_type': 'Google investigation'}

We can get the reasons for each ad disapproval and the dates when they were last shown for each region where they appeared.

**OC15: Does the platform indicate whether ad creatives were generated using artificial intelligence?**

*This item verifies whether the platform flags ad campaigns in which AI played a role in producing the content. The assessment should review ad records to confirm the presence of a field or label indicating the use of AI in ad production.*
 


In [123]:
df_creative_stats.columns

Index(['advertiser_id', 'creative_id', 'creative_page_url', 'ad_format_type',
       'advertiser_disclosed_name', 'advertiser_legal_name',
       'advertiser_location', 'advertiser_verification_status', 'region_stats',
       'audience_selection_approach_info', 'topic',
       'is_funded_by_google_ad_grants', 'ad_funded_by'],
      dtype='object')

In [125]:
df_removed_creative_stats.columns

Index(['creative_page_url', 'region_stats', 'audience_selection_approach_info',
       'disapproval'],
      dtype='object')

There are no fields regarding A.I. usage for creating or modifying ads

### CONSISTENCY

**OC23: Does the data retrieved by the API reflect what is displayed on the platform’s ad repository GUI?**

*This item verifies whether the data returned by the platform’s ad repository API corresponds to the information displayed on its GUI in all its levels of detail. It should be possible to identify in the API response information such as authorship, complete content, and other campaigning information (e.g., amount spent, impressions reached). The assessment should compare API responses with the GUI to confirm that key elements—such as authorship, full content, and campaign information (e.g., spending, impressions)—are consistently included.*


In [14]:
sql_query_specific_ad = f'''
SELECT
  CS.*
FROM
  bigquery-public-data.google_ads_transparency_center.creative_stats AS CS
WHERE
  CS.creative_id = 'CR03457172953559465985'
LIMIT
  100;
'''

df_specific_ad = query_api(project_id=PROJECT_ID, select=sql_query_specific_ad)

Downloading: 100%|█████████████████████████████████████████████████████████████|


In [17]:
df_specific_ad.loc[0]

advertiser_id                                                  AR02299254993238097921
creative_id                                                    CR03457172953559465985
creative_page_url                   https://adstransparency.google.com/advertiser/...
ad_format_type                                                                  VIDEO
advertiser_disclosed_name                                                  Centeia SL
advertiser_legal_name                                                      Centeia SL
advertiser_location                                                                AD
advertiser_verification_status                                               VERIFIED
region_stats                        [{'region_code': 'EEA', 'first_shown': '2025-1...
audience_selection_approach_info    {'demographic_info': 'CRITERIA_INCLUDED', 'geo...
topic                                                           Business & Industrial
is_funded_by_google_ad_grants                         

In [21]:
df_specific_ad.loc[0].creative_page_url

'https://adstransparency.google.com/advertiser/AR02299254993238097921/creative/CR03457172953559465985'

In [26]:
df_specific_ad.loc[0].region_stats[1]

{'region_code': 'FR',
 'first_shown': '2025-11-18',
 'last_shown': '2025-11-24',
 'times_shown_end_date': None,
 'times_shown_lower_bound': None,
 'times_shown_upper_bound': None,
 'times_shown_start_date': None,
 'times_shown_availability_date': '2026-02-16',
 'surface_serving_stats': None}

In [27]:
df_specific_ad.loc[0].audience_selection_approach_info

{'demographic_info': 'CRITERIA_INCLUDED',
 'geo_location': 'CRITERIA_INCLUDED',
 'contextual_signals': 'CRITERIA_INCLUDED',
 'customer_lists': 'CRITERIA_UNUSED',
 'topics_of_interest': 'CRITERIA_UNUSED'}

Most of the data retrieved from the BigQuery Google Ads tables seem to reflect what is displayed on the GUI. The exception is the content of the ad itself, which is only available on the GUI.

**OC24: Are the results returned by the platform consistently reproducible?**


*This item verifies whether data accessed and extracted via the platform’s ad repository at a given time is consistent with other collections performed similarly, including cases where content was deleted in the interim. The assessment should perform repeated queries to confirm reproducibility of results.*

Test instructions: Please, develop a test that collects ads for 5 times using the same request parameters and the same endpoint. Save each response in separate files.

In [6]:
sql_query_repeated = f"""
SELECT
  CS.advertiser_id,
  CS.creative_id,
  CS.creative_page_url,
  CS.ad_format_type,
  CS.advertiser_disclosed_name,
  CS.advertiser_legal_name 	
FROM
  {creative_stats_tb} AS CS
CROSS JOIN
  UNNEST(CS.region_stats) AS RS
WHERE
  RS.region_code = "FR"
LIMIT
  100;
"""

total_runs = 5 
data_dir = "../../data"
for idx in range(total_runs):
    index = idx + 1
    filename = f"eu-google-ads-question-24-run-{index}-{datetime.now().isoformat()}.json"

    response = query_api(project_id=PROJECT_ID, select=sql_query_repeated)
    response.to_json(f'{data_dir}/{filename}', orient='records')

Downloading: 100%|█████████████████████████████████████████████████████████████|
Downloading: 100%|█████████████████████████████████████████████████████████████|
Downloading: 100%|█████████████████████████████████████████████████████████████|
Downloading: 100%|█████████████████████████████████████████████████████████████|
Downloading: 100%|█████████████████████████████████████████████████████████████|


In [15]:
filenames = [
    'eu-google-ads-question-24-run-1-2025-12-14T17:17:13.853473.json',
    'eu-google-ads-question-24-run-2-2025-12-14T17:17:17.494718.json',
    'eu-google-ads-question-24-run-3-2025-12-14T17:17:18.461305.json',
    'eu-google-ads-question-24-run-4-2025-12-14T17:17:19.245543.json',
    'eu-google-ads-question-24-run-5-2025-12-14T17:17:19.894691.json'
]
# results for the first run
df_1 = pd.read_json(f'{data_dir}/{filenames[0]}', orient='records')

# comparing with results from the other runs
[
    df_1.equals(
        pd.read_json(f'{data_dir}/{filename}', orient='records')
    )
    for filename in filenames[1:]
]

[True, True, True, True]

**OC25: Is the data returned by the platform consistent with the parameters and filters used in the request?**

*This item verifies whether the data retrieved through the ad repository accurately reflects the parameters and filters specified at the time of the request. The assessment should run test queries with different filters to confirm that results consistently match the requested conditions.*

Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files. 

In [29]:
sql_query_base = f"""
SELECT
  CS.advertiser_id,
  CS.creative_id,
  CS.creative_page_url,
  CS.ad_format_type,
  CS.advertiser_disclosed_name,
  CS.advertiser_legal_name,
  CS.advertiser_verification_status,
  CS.topic,
FROM
  {creative_stats_tb} AS CS
"""

sql_query_suffix = """
LIMIT
  10;
"""

filter_advertiser_id = """
WHERE
  CS.advertiser_id = 'AR10967695351116464129'
"""
filter_format_type = """
WHERE
  CS.ad_format_type = 'TEXT'
"""
filter_creative_id = """
WHERE
  CS.creative_id = 'CR07377182285994393601'
"""

filter_advertiser_verification_status = """
WHERE
  CS.advertiser_verification_status = 'UNVERIFIED'
"""

filter_topic = """
WHERE
  CS.topic = 'Health'
"""

filters = [
    {'name': 'advertiser_id', 'filter': filter_advertiser_id},
    {'name': 'format_type', 'filter': filter_format_type},
    {'name': 'creative_id', 'filter': filter_creative_id},
    {'name': 'advertiser_verification_status', 'filter': filter_advertiser_verification_status},
    {'name': 'topic', 'filter': filter_topic}
]

data_dir = "../../data"
for idx, entry in enumerate(filters):
    index = idx + 1 
    filename = f"eu-google-ads-question-25-run-{index}-{entry['name']}-{datetime.now().isoformat()}.json"
    
    response = query_api(project_id=PROJECT_ID, select=sql_query_base + entry['filter'] + sql_query_suffix)
    response.to_json(f'{data_dir}/{filename}', orient='records')

Downloading: 100%|█████████████████████████████████████████████████████████████|
Downloading: 100%|█████████████████████████████████████████████████████████████|
Downloading: 100%|█████████████████████████████████████████████████████████████|
Downloading: 100%|█████████████████████████████████████████████████████████████|
Downloading: 100%|█████████████████████████████████████████████████████████████|


The results were appropriate for each filter.

### RELEVANCE

**OC26: Does the platform allow the use of temporal filters to retrieve data on ads?**

*This item verifies whether the ad repository allows filtering ad campaign data based on the time period during which the ads were served. The assessment should test queries with temporal filters to confirm that results accurately reflect the specified date ranges.*



In [31]:
date_filter = '2025-01-01'

# Query for filtering ads where, regardless of region, each once shown ad was not shown from 2025-01-01 onwards.
sql_query_temporal_filter = f'''
  SELECT
  CS.advertiser_id,
  CS.creative_id,
  CS.creative_page_url,
  CS.advertiser_legal_name, 	
  CS.region_stats
FROM
  {creative_stats_tb} AS CS
WHERE
  NOT EXISTS ( -- Check if an element exists that violates the rule (i.e., happened in 2025)
    SELECT 1
    FROM CS.region_stats AS nested_region_stats
    WHERE PARSE_DATE('%Y-%m-%d' , nested_region_stats.last_shown) >= DATE '{date_filter}'
  )
LIMIT
  100;
'''

df_temporal_filter = query_api(project_id=PROJECT_ID, select=sql_query_temporal_filter)
df_temporal_filter

Downloading: 100%|█████████████████████████████████████████████████████████████|


,advertiser_id,creative_id,creative_page_url,advertiser_legal_name,region_stats
0,AR11966622590031626241,CR17669152700173910017,https://adstransparency.google.com/advertiser/...,,"[{'region_code': 'EEA', 'first_shown': '2024-1..."
1,AR12834952277340979201,CR03580935391131205633,https://adstransparency.google.com/advertiser/...,"VIVO VOJO, SL","[{'region_code': 'BE', 'first_shown': '2024-12..."
2,AR04975799086490845185,CR08605633573640208385,https://adstransparency.google.com/advertiser/...,,"[{'region_code': 'EEA', 'first_shown': '2024-1..."
3,AR09148953722632536065,CR04050065889960132609,https://adstransparency.google.com/advertiser/...,APPRICOT IT CONSULTANTS L.L.C - FZ,"[{'region_code': 'DE', 'first_shown': '2024-07..."
4,AR00012045441940062209,CR13556508002839691265,https://adstransparency.google.com/advertiser/...,AZ Intelligence DMCC,"[{'region_code': 'DE', 'first_shown': '2023-03..."
...,...,...,...,...,...
95,AR18416962152003796993,CR16252106581062189057,https://adstransparency.google.com/advertiser/...,Auslogics Labs Pty Ltd,"[{'region_code': 'EEA', 'first_shown': '2023-0..."
96,AR17248699436532498433,CR00493115125218148353,https://adstransparency.google.com/advertiser/...,CBD College Pty Ltd,"[{'region_code': 'DE', 'first_shown': '2024-03..."
97,AR00495148362736074753,CR00202836599422058497,https://adstransparency.google.com/advertiser/...,Cable Chick,"[{'region_code': 'AT', 'first_shown': '2023-03..."
98,AR01526592455451869185,CR09070332937529982977,https://adstransparency.google.com/advertiser/...,Candlefox Pty Ltd,"[{'region_code': 'EEA', 'first_shown': '2023-0..."


In [36]:
all_last_shown_dates_from_table = []
for region_stats in df_temporal_filter.region_stats:
    for region in region_stats:
        all_last_shown_dates_from_table.append(region['last_shown'])

max(all_last_shown_dates_from_table)

'2024-12-31'

**OC27: Does the platform allow filtering advertising data by ad category?**

*This item verifies whether the ad repository allows filtering ad campaign data based on any possible categories assigned at the time of ad creation. The assessment should run test queries with category filters to confirm that results align with the selected classifications.*


In [24]:
topic = 'Health'

sql_query_topic = f"""
SELECT
  CS.advertiser_id,
  CS.creative_id,
  CS.creative_page_url,
  CS.advertiser_legal_name,
  CS.topic
FROM
  {creative_stats_tb} AS CS
WHERE
  CS.topic = '{topic}'
LIMIT
  100;
"""

df_topic = query_api(project_id=PROJECT_ID, select=sql_query_topic)
df_topic

Downloading: 100%|█████████████████████████████████████████████████████████████|


,advertiser_id,creative_id,creative_page_url,advertiser_legal_name,topic
0,AR00332404114791071745,CR12070965573492670465,https://adstransparency.google.com/advertiser/...,ALLFORMANCE FZCO,Health
1,AR01536504037380194305,CR02854253175488118785,https://adstransparency.google.com/advertiser/...,Dmitrii Shumkov,Health
2,AR13318446985053732865,CR02501850765253935105,https://adstransparency.google.com/advertiser/...,Mohamed Al-Ali,Health
3,AR13435610634871898113,CR07278917866668163073,https://adstransparency.google.com/advertiser/...,TechJini LLC,Health
4,AR13141408158234705921,CR08907813402273906689,https://adstransparency.google.com/advertiser/...,ELITE DENTAL,Health
...,...,...,...,...,...
95,AR07373240313470517249,CR09113000310879551489,https://adstransparency.google.com/advertiser/...,The Synergist,Health
96,AR16804445096224227329,CR08792224716963184641,https://adstransparency.google.com/advertiser/...,Therese Kubli,Health
97,AR08944917693463527425,CR12292270875884912641,https://adstransparency.google.com/advertiser/...,"""КУАЙСЕР ФАРМА БЪЛГАРИЯ"" ЕООД",Health
98,AR00253587032503222273,CR17621878836960428033,https://adstransparency.google.com/advertiser/...,"""Ортодент-Д-р Цветелина Узунова-Индивидуална п...",Health


**OC28: Does the platform allow filtering advertising data by geographic location?**

*This item assesses whether the ad repository allows filtering data by one or more subnational geographic locations where the ads were served. The assessment should test queries with location filters to confirm that results match the specified areas.*

The only region field available on the Google Ads Transparency BigQuery tables works at the country level.

### ACCURACY

**OC29: Does the platform provide age and gender data on the audiences of ads?**

*This item verifies whether the platform provides data on the age and gender of audiences reached. The assessment should review the ad records to confirm that these breakdowns are available and consistently reported*

The BigQuery tables have no information regarding age and gender on the audiences of ads.

**OC30: Does the platform provide subnational geographic data on the audience reached by ads?**

*This item verifies whether the platform provides data on the subnational geographic location of audiences reached. The assessment should review the ad records to confirm that such location breakdowns are available and consistently reported.*

In [14]:
df_creative_stats.region_stats.loc[0][0]

{'region_code': 'DE',
 'first_shown': '2025-01-22',
 'last_shown': '2025-01-22',
 'times_shown_end_date': '2025-01-22',
 'times_shown_lower_bound': 0,
 'times_shown_upper_bound': 1000,
 'times_shown_start_date': '2025-01-22',
 'times_shown_availability_date': None,
 'surface_serving_stats': {'surface_serving_stats': array([{'surface': 'PLAY', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'MAPS', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'YOUTUBE', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'SHOPPING', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'SEARCH', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None}],
        dtype=

In [41]:
all_regions = set()
for region_stats in df_creative_stats.region_stats:
    for region in region_stats:
        all_regions.add(region['region_code'])
all_regions

{'AT',
 'BE',
 'BG',
 'CY',
 'CZ',
 'DE',
 'DK',
 'EE',
 'EEA',
 'ES',
 'FI',
 'FR',
 'GF',
 'GP',
 'GR',
 'HR',
 'HU',
 'IE',
 'IS',
 'IT',
 'LT',
 'LU',
 'LV',
 'MF',
 'MQ',
 'MT',
 'NL',
 'NO',
 'PL',
 'PT',
 'RE',
 'RO',
 'SE',
 'SI',
 'SK',
 'TR',
 'YT'}

The BigQuery tables only prodive region information on the country level and above (country codes and EEA acronyms).

## <<<<<<<<<<<<<<4-Digit Region Codes: You may encounter 4-digit codes (e.g., 2050, 2124) which are typically Google's internal criteria IDs for geographic locations. These are often mapped to specific cities, metropolitan areas, or other regions rather than just countries.

**OC31: Does the platform include data on audience targeting criteria defined by advertisers?**

*This item verifies whether the platform provides data on audience targeting criteria defined by the advertiser when publishing ads (e.g., demographic and geographic segments, as well as interests, attitudes, behaviors, and keywords). The assessment should review ad records to confirm that these targeting parameters are available and consistently reported.*


In [133]:
df_creative_stats.audience_selection_approach_info.loc[2]

{'demographic_info': 'CRITERIA_INCLUDED',
 'geo_location': 'CRITERIA_INCLUDED',
 'contextual_signals': 'CRITERIA_INCLUDED',
 'customer_lists': 'CRITERIA_UNUSED',
 'topics_of_interest': 'CRITERIA_UNUSED'}

The platform signals the inclusion and exclusion of filters for audience targeting, but only with broad definitions of types of criteria, without more specific information for each type.

**OC32: Does the platform provide granular volume ranges for ad impressions?**

*This item verifies whether the ad repository provides impression values for ads, using ranges that closely approximate the actual numbers. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to 1,000 impressions should be displayed in intervals no larger than 100; between 1,000 and 10,000 in intervals no larger than 1,000; between 10,000 and 100,000 in intervals no larger than 10,000; between 100,000 and 1 million or above, in intervals no larger than 100,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*


In [160]:
d = df_creative_stats.region_stats.loc[2][1]
{
    'lower_bound': d['times_shown_lower_bound'],
    'upper_bound': d['times_shown_upper_bound'],
    'difference': d['times_shown_upper_bound'] - d['times_shown_lower_bound']
}

{'lower_bound': 1000, 'upper_bound': 2000, 'difference': 1000}

In [161]:
d = df_creative_stats.region_stats.loc[29][0]
{
    'lower_bound': d['times_shown_lower_bound'],
    'upper_bound': d['times_shown_upper_bound'],
    'difference': d['times_shown_upper_bound'] - d['times_shown_lower_bound']
}

{'lower_bound': 15000, 'upper_bound': 20000, 'difference': 5000}

In [162]:
d = df_creative_stats.region_stats.loc[8][3]
{
    'lower_bound': d['times_shown_lower_bound'],
    'upper_bound': d['times_shown_upper_bound'],
    'difference': d['times_shown_upper_bound'] - d['times_shown_lower_bound']
}

{'lower_bound': 0, 'upper_bound': 1000, 'difference': 1000}

As displayed by the output of the above cell, when the upper bound is 1000, the lower bound was 0. Thus, the dataset contains intervals higher than the required ammount for values up to 1000.

**OC33: Does the platform provide granular investment ranges for ad spending?**

*This item verifies whether the ad repository provides spending values for ads, using ranges that closely approximate the actual amounts. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to $100 should be displayed in intervals no larger than $10; between $100 and $1,000 in intervals no larger than $100; and between $1,000 and $10,000 in intervals no larger than $1,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*


There are no fields regarding investment for ad spending